In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 05 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 05.C1 — Generate all nonempty faces

Given distinct integer vertices of one simplex, return a set of its nonempty face tuples with vertices sorted.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The combinations function enumerates subsets of a requested size.

</details>

In [3]:
def simplex_faces(vertices):
    v=tuple(sorted(vertices))
    return {face for size in range(1,len(v)+1) for face in combinations(v,size)}

In [4]:
assert len(simplex_faces([2,0,1]))==7
assert (0,2) in simplex_faces([2,0,1])
assert len(simplex_faces([0,1,2,3]))==15
print("Triangle and tetrahedron nonempty face counts: 7 and 15")

Triangle and tetrahedron nonempty face counts: 7 and 15


**Why this works and what it does not establish:** An abstract triangle consists of a three-vertex simplex plus all its faces. The empty face is conventionally implicit in some texts; this course does not store it in code.

## 05.C2 — Count simplices by dimension

Return counts for dimensions 0 through maximum dimension in a nonempty complex. A tuple with r vertices has dimension r-1.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Count tuple lengths, not coordinate columns.

</details>

In [5]:
def simplex_counts(complex_):
    largest=max(len(s)-1 for s in complex_)
    return [sum(len(s)==k+1 for s in complex_) for k in range(largest+1)]

In [6]:
K=closure([(0,1,2)])
assert simplex_counts(K)==[3,3,1]
print("Filled triangle V,E,F:",simplex_counts(K))

Filled triangle V,E,F: [3, 3, 1]


**Why this works and what it does not establish:** The dimension of a simplex is intrinsic to its vertex count. A triangle can be drawn in a two-dimensional page or embedded in three-dimensional space without changing its simplicial dimension.

## 05.C3 — Check the face condition

Return whether every nonempty face of each listed simplex also occurs. Assume each supplied tuple already has distinct sorted vertices.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

For each simplex, enumerate the subsets that its existence requires.

</details>

In [7]:
def has_all_faces(complex_):
    present=set(complex_)
    return all(face in present for s in present for size in range(1,len(s)+1) for face in combinations(s,size))

In [8]:
assert has_all_faces(closure([(0,1,2)]))
assert not has_all_faces([(0,),(1,),(0,1,2)])
print("Missing-face example rejected")

Missing-face example rejected


**Why this works and what it does not establish:** This question only checks face closure; duplicate records and invalid vertex tuples are separate validation questions handled by the production course helper.

## 05.C4 — Compute Euler characteristic

For a finite complex return the alternating sum of simplex counts: vertices minus edges plus triangles and so on.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Even dimensions contribute positively; odd dimensions contribute negatively.

</details>

In [9]:
def euler_from_simplices(complex_):
    return sum((-1)**(len(s)-1) for s in complex_)

In [10]:
boundary=closure([(0,1),(1,2),(0,2)])
filled=closure([(0,1,2)])
assert euler_from_simplices(boundary)==0
assert euler_from_simplices(filled)==1
print("Outline versus filled triangle Euler values: 0 versus 1")

Outline versus filled triangle Euler values: 0 versus 1


**Why this works and what it does not establish:** The alternating count is useful but incomplete. The same Euler characteristic does not imply the same homology groups, much less homeomorphism.

## 05.C5 — Build a real image threshold object

Return simplices in the declared closed-pixel model whose entrance value 1-intensity/16 is <= threshold. Use pixel_filtration, keeping all included faces.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The filtration entries already assign shared faces their earliest valid time.

</details>

In [11]:
def image_threshold_complex(image,threshold):
    f=pixel_filtration(1-np.asarray(image,dtype=float)/16)
    return [simplex for simplex,value in f if value<=threshold]

In [12]:
image,sample_id=digit_example(0)
K=image_threshold_complex(image,.5)
validate_complex(K)
assert len(K)>0
print("Training image",sample_id,"threshold complex simplices:",len(K))

Training image 0 threshold complex simplices: 172


**Why this works and what it does not establish:** The object is a union of closed squares, with corner-touching pixels connected, subdivided into triangles. This is not the same as every possible image-connectivity convention.